In [1]:
import QuantLib as ql

In [2]:
today = ql.Date(2, ql.May, 2019)
ql.Settings.instance().evaluationDate = today

QuantLib defines different kinds of interest-rate term structures. Depending on your choice of model or the data you have available, they can model discount factors, instantaneous forward rates or zero rates. Their implementation can be, for instance, a simple one returning a constant value for the forward rate…

In [3]:
curve_1 = ql.FlatForward(today, 0.01, ql.Actual365Fixed())

…or one interpolating between a number of precalculated discount factors for a set of given dates.

More complex curves are also available (e.g., ones that can be bootstrapped or fitted over a set of quotes) and will be described in more detail in other notebooks.

<h2>A common interface</h2>

Because of the way they’re defined, it might look like the first curve above is supposed to be used for forecasting and the second curve for discounting, and indeed, they can be used this way. I can pass the first curve to an instance of an interest-rate index, allowing it to forecast its future fixings…

In [4]:
forecast_curve = ql.RelinkableYieldTermStructureHandle(curve_1)
index_6M = ql.Euribor6M(forecast_curve)
print(index_6M.fixing(ql.Date(1, ql.July, 2019)))



0.009887915724457295


…and I can pass the second to an engine that will use it to discount a sequence of bond cashflows.

In [5]:
schedule = ql.Schedule(
    ql.Date(8, ql.February, 2013),
    ql.Date(8, ql.February, 2023),
    ql.Period(6, ql.Months),
    ql.TARGET(),
    ql.Following,
    ql.Following,
    ql.DateGeneration.Backward,
    False
)

bond = ql.FixedRateBond(
    3, 100.0, schedule, [0.03], ql.Thirty360(ql.Thirty360.BondBasis)
)

discount_curve = ql.RelinkableYieldTermStructureHandle(curve_2)
bond.setPricingEngine(ql.DiscountingBondEngine(discount_curve))
print(bond.cleanPrice())

NameError: name 'curve_2' is not defined